In [ ]:
# Setup — run once per session
!pip install -q timm einops ml-collections medpy SimpleITK tensorboardX thop

!cp -r /kaggle/input/datasets/deepsotaai/adada-transunet-code/DA-TransUNet /kaggle/working/DA-TransUNet
!cp -r /kaggle/input/datasets/deepsotaai/vit-pretrained-weights/model       /kaggle/working/model

# Kaggle strips '+' from filenames — rename back (|| true silences error if already correct)
!mv /kaggle/working/model/vit_checkpoint/imagenet21k/R50ViT-B_16.npz \
    /kaggle/working/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz 2>/dev/null || true

# Prevent HuggingFace 'datasets' library from shadowing local datasets/ folder
!touch /kaggle/working/DA-TransUNet/datasets/__init__.py

# Symlink Synapse data (train.py hardcodes ../data/Synapse/ and ignores --root_path)
!mkdir -p /kaggle/working/data/Synapse
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz  /kaggle/working/data/Synapse/train_npz
!ln -sfn /kaggle/input/datasets/dogcdt/synapse/Synapse/test_vol_h5 /kaggle/working/data/Synapse/test_vol_h5

print('Setup complete.')

In [ ]:
%%bash
echo "========================================"
echo " DA-TransUNet  |  TRAINING"
echo " Started: $(date)"
echo "========================================"
cd /kaggle/working/DA-TransUNet
python -u train.py \
  --dataset      Synapse \
  --vit_name     R50-ViT-B_16 \
  --max_epochs   150 \
  --batch_size   24 \
  --base_lr      0.01 \
  --n_skip       3 \
  --img_size     224 \
  --seed         1234 \
  --val_interval 10
echo "========================================"
echo " Training finished: $(date)"
echo "========================================"

In [ ]:
%%bash
echo "========================================"
echo " DA-TransUNet  |  INFERENCE"
echo " Started: $(date)"
echo "========================================"
cd /kaggle/working/DA-TransUNet
python -u test.py \
  --dataset    Synapse \
  --vit_name   R50-ViT-B_16 \
  --num_classes 9 \
  --img_size   224 \
  --is_savenii
echo "========================================"
echo " Inference finished: $(date)"
echo "========================================"